# PART 1 — SPREADSHEET CLEANING AND BUSINESS LOGIC

In [2]:
import pandas as pd
import numpy as np
import json
import os

# Set paths
data_dir = '../01_data'
raw_dir = os.path.join(data_dir, 'raw')
processed_dir = os.path.join(data_dir, 'processed')

# Ensure processed dir exists
os.makedirs(processed_dir, exist_ok=True)

In [3]:
# Load data
transactions = pd.read_csv(os.path.join(raw_dir, 'transactions_raw.csv'))
merchants = pd.read_csv(os.path.join(raw_dir, 'merchant_master.csv'))
exchange_rates = pd.read_csv(os.path.join(raw_dir, 'exchange_rates.csv'))

print("Transactions shape:", transactions.shape)
print("Merchants shape:", merchants.shape)
print("Exchange rates shape:", exchange_rates.shape)

Transactions shape: (30, 10)
Merchants shape: (5, 5)
Exchange rates shape: (18, 3)


In [4]:
# Clean transactions
transactions['merchant_name'] = transactions['merchant_name'].str.strip().str.title()
transactions['status'] = transactions['status'].str.lower().str.strip()

# Standardize status
status_map = {
    'captured': 'captured',
    'failed e05 timeout': 'failed',
    'chargeback': 'chargeback'
}
transactions['status'] = transactions['status'].map(status_map).fillna('captured')  # assume others are captured

# Clean risk_score
def clean_risk_score(x):
    if pd.isna(x):
        return 0
    x = str(x).replace('score:', '').replace('risk-', '')
    try:
        return int(x)
    except:
        return 0

transactions['risk_score'] = transactions['risk_score'].apply(clean_risk_score)

# Clean gateway_region
transactions['gateway_region'] = transactions['gateway_region'].str.strip().str.upper()

# Standardize date
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

print("Cleaned transactions head:")
print(transactions.head())

Cleaned transactions head:
  transaction_id transaction_date merchant_name  raw_amount currency  \
0           T001       2026-03-01    Alpha Mart      420000      INR   
1           T002       2026-03-01    Alpha Mart      210000      INR   
2           T003       2026-03-01   Beta Stores      510000      INR   
3           T004       2026-03-02  Beta  Stores      160000      INR   
4           T005       2026-03-02    Alpha Mart      390000      INR   

     status  risk_score gateway_region user_id payment_method  
0  captured          62           APAC    U001            UPI  
1  captured          55            NaN    U002           Card  
2  captured          71           APAC    U003     NetBanking  
3    failed          68           APAC    U004           Card  
4  captured          58            NaN    U001            UPI  


In [5]:
# Convert amounts to USD
exchange_rates['rate_date'] = pd.to_datetime(exchange_rates['rate_date'])
transactions = transactions.merge(exchange_rates, left_on=['transaction_date', 'currency'], right_on=['rate_date', 'currency'], how='left')
transactions['amount_usd'] = transactions['raw_amount'] * transactions['usd_rate']

# Drop extra columns
transactions.drop(['rate_date', 'usd_rate'], axis=1, inplace=True)

print("After conversion:")
print(transactions[['transaction_id', 'raw_amount', 'currency', 'amount_usd']].head())

After conversion:
  transaction_id  raw_amount currency  amount_usd
0           T001      420000      INR      4998.0
1           T002      210000      INR      2499.0
2           T003      510000      INR      6069.0
3           T004      160000      INR      1920.0
4           T005      390000      INR      4680.0


In [6]:
# Enrich with merchant data
merchants['merchant_name'] = merchants['merchant_name'].str.strip().str.title()
transactions = transactions.merge(merchants, on='merchant_name', how='left')

print("Enriched transactions:")
print(transactions.head())

Enriched transactions:
  transaction_id transaction_date merchant_name  raw_amount currency  \
0           T001       2026-03-01    Alpha Mart      420000      INR   
1           T002       2026-03-01    Alpha Mart      210000      INR   
2           T003       2026-03-01   Beta Stores      510000      INR   
3           T004       2026-03-02  Beta  Stores      160000      INR   
4           T005       2026-03-02    Alpha Mart      390000      INR   

     status  risk_score gateway_region user_id payment_method  amount_usd  \
0  captured          62           APAC    U001            UPI      4998.0   
1  captured          55            NaN    U002           Card      2499.0   
2  captured          71           APAC    U003     NetBanking      6069.0   
3    failed          68           APAC    U004           Card      1920.0   
4  captured          58            NaN    U001            UPI      4680.0   

  merchant_id account_manager merchant_category default_region  
0        M001   

In [8]:
# Save cleaned transactions
transactions.to_csv(os.path.join(processed_dir, 'cleaned_transactions.csv'), index=False)

# Create merchant risk summary
merchant_summary = transactions.groupby('merchant_name').agg(
    total_transactions=('transaction_id', 'count'),
    avg_risk_score=('risk_score', 'mean'),
    total_gmv=('amount_usd', 'sum'),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()

merchant_summary.to_csv(os.path.join(processed_dir, 'merchant_risk_summary.csv'), index=False)

print("Files saved:")
print("- cleaned_transactions.csv")
print("- merchant_risk_summary.csv")

Files saved:
- cleaned_transactions.csv
- merchant_risk_summary.csv


In [9]:
# Calculate answers for spreadsheet_answers.md
total_raw_rows = len(pd.read_csv(os.path.join(raw_dir, 'transactions_raw.csv')))  # reload to get raw count
total_cleaned_rows = len(transactions)
invalid_missing_rows = transactions.isnull().sum().sum()  # total nulls

top_region_gmv = transactions.groupby('gateway_region')['amount_usd'].sum().idxmax()
num_high_value = transactions['high_value_flag'].sum()
num_high_risk = transactions['high_risk_flag'].sum()

captured = transactions[transactions['status'] == 'captured']
top_merchant_gmv = captured.groupby('merchant_name')['amount_usd'].sum().idxmax()

print(f"Total raw rows: {total_raw_rows}")
print(f"Total cleaned rows: {total_cleaned_rows}")
print(f"Invalid/missing rows handled: {invalid_missing_rows}")
print(f"Top region by GMV: {top_region_gmv}")
print(f"Number of high value transactions: {num_high_value}")
print(f"Number of high risk transactions: {num_high_risk}")
print(f"Top merchant by captured GMV: {top_merchant_gmv}")

Total raw rows: 30
Total cleaned rows: 30
Invalid/missing rows handled: 45
Top region by GMV: APAC
Number of high value transactions: 6
Number of high risk transactions: 9
Top merchant by captured GMV: Alpha Mart


# PART 2 — SQL ANALYSIS

In [11]:
# Q1: Count transactions by status
q1 = transactions.groupby('status').size().reset_index(name='count')
print("Q1:")
print(q1.to_string())

# Q2: Total captured GMV by merchant
q2 = transactions[transactions['status'] == 'captured'].groupby('merchant_name')['amount_usd'].sum().reset_index(name='total_captured_gmv')
print("\nQ2:")
print(q2.to_string())

# Q3: Top 10 merchants by captured GMV
q3 = q2.sort_values('total_captured_gmv', ascending=False).head(10)
print("\nQ3:")
print(q3.to_string())

# Q4: Daily GMV and successful transaction count
q4 = transactions[transactions['status'] == 'captured'].groupby('transaction_date').agg(
    daily_gmv=('amount_usd', 'sum'),
    successful_count=('transaction_id', 'count')
).reset_index()
print("\nQ4:")
print(q4.to_string())

# Q5: Merchants with chargeback ratio > 1%
chargeback_counts = transactions.groupby('merchant_name').agg(
    total=('transaction_id', 'count'),
    chargebacks=('status', lambda x: (x == 'chargeback').sum())
)
chargeback_counts['ratio'] = chargeback_counts['chargebacks'] / chargeback_counts['total']
q5 = chargeback_counts[chargeback_counts['ratio'] > 0.01].reset_index()[['merchant_name', 'ratio']]
print("\nQ5:")
print(q5.to_string())

# Q6: Regions with avg risk >50 and >20 transactions
q6 = transactions.groupby('gateway_region').agg(
    avg_risk=('risk_score', 'mean'),
    transaction_count=('transaction_id', 'count')
)
q6 = q6[(q6['avg_risk'] > 50) & (q6['transaction_count'] > 20)].reset_index()
print("\nQ6:")
print(q6.to_string())

# Q7: Users with 3+ failed/chargeback on same day
failed_chargeback = transactions[transactions['status'].isin(['failed', 'chargeback'])]
q7 = failed_chargeback.groupby(['user_id', 'transaction_date']).size().reset_index(name='count')
q7 = q7[q7['count'] >= 3]['user_id'].unique()
print("\nQ7:")
print(list(q7))

# Q8: Chargeback count, unique users, amount by merchant
q8 = transactions[transactions['status'] == 'chargeback'].groupby('merchant_name').agg(
    chargeback_count=('transaction_id', 'count'),
    unique_affected_users=('user_id', 'nunique'),
    chargeback_amount=('amount_usd', 'sum')
).reset_index()
print("\nQ8:")
print(q8.to_string())

Q1:
       status  count
0    captured     19
1  chargeback      4
2      failed      7

Q2:
   merchant_name  total_captured_gmv
0     Alpha Mart             29984.5
1   Beta  Stores             11940.0
2    Beta Stores             21491.0
3    City Pharma              8640.0
4  Delta Travels             10300.0

Q3:
   merchant_name  total_captured_gmv
0     Alpha Mart             29984.5
2    Beta Stores             21491.0
1   Beta  Stores             11940.0
4  Delta Travels             10300.0
3    City Pharma              8640.0

Q4:
  transaction_date  daily_gmv  successful_count
0       2026-03-01    26382.0                 5
1       2026-03-02    11080.0                 3
2       2026-03-03    16031.5                 4
3       2026-03-04    13920.0                 4
4       2026-03-05     6136.0                 1
5       2026-03-06     8806.0                 2

Q5:
   merchant_name     ratio
0    Alpha  Mart  0.333333
1    Beta Stores  0.200000
2  Delta Travels  0.250000
3   

In [12]:
# Create sql_answers.md
sql_answers = f"""# SQL Answers

## Q1
### Query
SELECT status, COUNT(*) as count FROM cleaned_transactions GROUP BY status;
### Result Summary
{q1.to_string(index=False)}

## Q2
### Query
SELECT merchant_name, SUM(amount_usd) as total_captured_gmv FROM cleaned_transactions WHERE status = 'captured' GROUP BY merchant_name;
### Result Summary
{q2.to_string(index=False)}

## Q3
### Query
SELECT merchant_name, SUM(amount_usd) as total_captured_gmv FROM cleaned_transactions WHERE status = 'captured' GROUP BY merchant_name ORDER BY total_captured_gmv DESC LIMIT 10;
### Result Summary
{q3.to_string(index=False)}

## Q4
### Query
SELECT transaction_date, SUM(amount_usd) as daily_gmv, COUNT(*) as successful_count FROM cleaned_transactions WHERE status = 'captured' GROUP BY transaction_date ORDER BY transaction_date;
### Result Summary
{q4.to_string(index=False)}

## Q5
### Query
SELECT merchant_name, SUM(CASE WHEN status = 'chargeback' THEN 1 ELSE 0 END) * 1.0 / COUNT(*) as chargeback_ratio FROM cleaned_transactions GROUP BY merchant_name HAVING chargeback_ratio > 0.01;
### Result Summary
{q5.to_string(index=False)}

## Q6
### Query
SELECT gateway_region, AVG(risk_score) as avg_risk, COUNT(*) as transaction_count FROM cleaned_transactions GROUP BY gateway_region HAVING avg_risk > 50 AND transaction_count > 20;
### Result Summary
{q6.to_string(index=False)}

## Q7
### Query
SELECT user_id FROM cleaned_transactions WHERE status IN ('failed', 'chargeback') GROUP BY user_id, DATE(transaction_date) HAVING COUNT(*) >= 3;
### Result Summary
{', '.join(q7)}

## Q8
### Query
SELECT merchant_name, SUM(CASE WHEN status = 'chargeback' THEN 1 ELSE 0 END) as chargeback_count, COUNT(DISTINCT user_id) as unique_affected_users, SUM(CASE WHEN status = 'chargeback' THEN amount_usd ELSE 0 END) as chargeback_amount FROM cleaned_transactions GROUP BY merchant_name;
### Result Summary
{q8.to_string(index=False)}
"""

with open('../03_sql/sql_answers.md', 'w') as f:
    f.write(sql_answers)

print("sql_answers.md created")

sql_answers.md created


# PART 3 — PYTHON RECONCILIATION WORKFLOW

In [13]:
# Load ledger and gateway
ledger = pd.read_csv(os.path.join(raw_dir, 'ledger.csv'))
gateway = pd.read_csv(os.path.join(raw_dir, 'gateway.csv'))

print("Ledger shape:", ledger.shape)
print("Gateway shape:", gateway.shape)

# Check duplicates and nulls
print("Ledger duplicates:", ledger.duplicated().sum())
print("Gateway duplicates:", gateway.duplicated().sum())
print("Ledger nulls:", ledger.isnull().sum().sum())
print("Gateway nulls:", gateway.isnull().sum().sum())

Ledger shape: (10, 6)
Gateway shape: (9, 6)
Ledger duplicates: 0
Gateway duplicates: 0
Ledger nulls: 0
Gateway nulls: 0


In [14]:
# Identify missing records
merged = ledger.merge(gateway, on='transaction_id', how='outer', indicator=True, suffixes=('_ledger', '_gateway'))

missing_in_gateway = merged[merged['_merge'] == 'left_only'][['transaction_id', 'transaction_date_ledger', 'merchant_id_ledger', 'amount_usd_ledger', 'status_ledger', 'payment_method_ledger']]
missing_in_gateway.columns = ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']
missing_in_gateway.to_csv(os.path.join(processed_dir, 'missing_in_gateway.csv'), index=False)

missing_in_ledger = merged[merged['_merge'] == 'right_only'][['transaction_id', 'transaction_date_gateway', 'merchant_id_gateway', 'amount_usd_gateway', 'status_gateway', 'payment_method_gateway']]
missing_in_ledger.columns = ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']
missing_in_ledger.to_csv(os.path.join(processed_dir, 'missing_in_ledger.csv'), index=False)

print("Missing in gateway:", len(missing_in_gateway))
print("Missing in ledger:", len(missing_in_ledger))

Missing in gateway: 2
Missing in ledger: 1


In [15]:
# Identify mismatches in common records
common = merged[merged['_merge'] == 'both']

amount_mismatches = common[common['amount_usd_ledger'] != common['amount_usd_gateway']][['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']]
amount_mismatches.columns = ['transaction_id', 'ledger_amount', 'gateway_amount']
amount_mismatches.to_csv(os.path.join(processed_dir, 'amount_mismatches.csv'), index=False)

status_mismatches = common[common['status_ledger'] != common['status_gateway']][['transaction_id', 'status_ledger', 'status_gateway']]
status_mismatches.columns = ['transaction_id', 'ledger_status', 'gateway_status']
status_mismatches.to_csv(os.path.join(processed_dir, 'status_mismatches.csv'), index=False)

print("Amount mismatches:", len(amount_mismatches))
print("Status mismatches:", len(status_mismatches))

Amount mismatches: 2
Status mismatches: 1


In [16]:
# Build reconciliation report
reconciliation = merged.copy()
reconciliation['missing_in_gateway'] = (reconciliation['_merge'] == 'left_only').astype(int)
reconciliation['missing_in_ledger'] = (reconciliation['_merge'] == 'right_only').astype(int)
reconciliation['amount_mismatch'] = ((reconciliation['_merge'] == 'both') & (reconciliation['amount_usd_ledger'] != reconciliation['amount_usd_gateway'])).astype(int)
reconciliation['status_mismatch'] = ((reconciliation['_merge'] == 'both') & (reconciliation['status_ledger'] != reconciliation['status_gateway'])).astype(int)

# Select columns
reconciliation_report = reconciliation[['transaction_id', 'missing_in_gateway', 'missing_in_ledger', 'amount_mismatch', 'status_mismatch']]
reconciliation_report.to_csv(os.path.join(processed_dir, 'reconciliation_report.csv'), index=False)

print("Reconciliation report shape:", reconciliation_report.shape)

Reconciliation report shape: (11, 5)


In [18]:
# Generate summary metrics
summary_metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(reconciliation_report[['missing_in_gateway', 'missing_in_ledger', 'amount_mismatch', 'status_mismatch']].sum().sum()),
    "ledger_total_amount": float(ledger['amount_usd'].sum()),
    "gateway_total_amount": float(gateway['amount_usd'].sum()),
    "amount_at_risk": float(amount_mismatches['ledger_amount'].sum() - amount_mismatches['gateway_amount'].sum())
}

with open(os.path.join('../04_python', 'summary_metrics.json'), 'w') as f:
    json.dump(summary_metrics, f, indent=4)

print("Summary metrics:", summary_metrics)

Summary metrics: {'total_ledger_rows': 10, 'total_gateway_rows': 9, 'missing_in_gateway_count': 2, 'missing_in_ledger_count': 1, 'amount_mismatch_count': 2, 'status_mismatch_count': 1, 'reconciliation_issue_count': 6, 'ledger_total_amount': 23340.0, 'gateway_total_amount': 20550.0, 'amount_at_risk': -10.0}


# PART 4 — JSON NORMALIZATION

In [19]:
# Read JSON
with open(os.path.join(raw_dir, 'api_response_sample.json'), 'r') as f:
    api_data = json.load(f)

# Flatten
rows = []
for batch in api_data['batches']:
    merchant = batch['merchant']
    for settlement in batch['settlements']:
        row = {
            'batch_id': batch['batch_id'],
            'merchant_id': merchant['merchant_id'],
            'merchant_name': merchant['merchant_name'],
            'region': merchant['region'],
            'settlement_id': settlement['settlement_id'],
            'amount_usd': settlement['amount_usd'],
            'status': settlement['status'],
            'processed_at': settlement['processed_at'],
            'bank_name': settlement['bank']['name'],
            'bank_country': settlement['bank']['country']
        }
        rows.append(row)

api_normalized = pd.DataFrame(rows)

# Convert date/time
api_normalized['processed_at'] = pd.to_datetime(api_normalized['processed_at'])

# Clean column names (already clean)

api_normalized.to_csv(os.path.join(processed_dir, 'api_normalized.csv'), index=False)

print("API normalized shape:", api_normalized.shape)
print(api_normalized.head())

API normalized shape: (6, 10)
  batch_id merchant_id  merchant_name region settlement_id  amount_usd  \
0     B001        M001     Alpha Mart   APAC          S001      1520.5   
1     B001        M001     Alpha Mart   APAC          S002       980.0   
2     B001        M001     Alpha Mart   APAC          S003       640.0   
3     B002        M004  Delta Travels     US          S004      2100.0   
4     B002        M004  Delta Travels     US          S005       500.0   

    status              processed_at bank_name bank_country  
0  settled 2026-03-07 08:10:00+00:00    Bank A           IN  
1  pending 2026-03-07 08:45:00+00:00    Bank A           IN  
2  settled 2026-03-07 09:15:00+00:00    Bank B           SG  
3  settled 2026-03-07 08:20:00+00:00    Bank C           US  
4   failed 2026-03-07 08:50:00+00:00    Bank C           US  


# PART 5 — DASHBOARD OUTPUT FILES

In [20]:
# Generate dashboard outputs
# Daily summary
daily_summary = transactions.groupby('transaction_date').agg(
    total_gmv=('amount_usd', 'sum'),
    confirmed_gmv=('amount_usd', lambda x: x[transactions.loc[x.index, 'status'] == 'captured'].sum()),
    transaction_count=('transaction_id', 'count'),
    success_count=('status', lambda x: (x == 'captured').sum())
).reset_index()
daily_summary['success_rate'] = daily_summary['success_count'] / daily_summary['transaction_count']
daily_summary['amount_at_risk'] = daily_summary['total_gmv'] - daily_summary['confirmed_gmv']
daily_summary.to_csv(os.path.join(processed_dir, 'daily_summary.csv'), index=False)

# Payment method breakdown
payment_method_breakdown = transactions.groupby('payment_method').agg(
    total_gmv=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
payment_method_breakdown.to_csv(os.path.join(processed_dir, 'payment_method_breakdown.csv'), index=False)

# Region breakdown
region_breakdown = transactions.groupby('gateway_region').agg(
    total_gmv=('amount_usd', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()
region_breakdown.to_csv(os.path.join(processed_dir, 'region_breakdown.csv'), index=False)

# Merchant performance summary
merchant_performance = transactions.groupby('merchant_name').agg(
    total_gmv=('amount_usd', 'sum'),
    confirmed_gmv=('amount_usd', lambda x: x[transactions.loc[x.index, 'status'] == 'captured'].sum()),
    transaction_count=('transaction_id', 'count'),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()
merchant_performance['success_rate'] = (merchant_performance['confirmed_gmv'] / merchant_performance['total_gmv']).fillna(0)
merchant_performance.to_csv(os.path.join(processed_dir, 'merchant_performance_summary.csv'), index=False)

print("Dashboard files generated.")

Dashboard files generated.


In [1]:
# Create spreadsheet workbook
cleaned = pd.read_csv('../01_data/processed/cleaned_transactions.csv')
cleaned.to_excel('../02_spreadsheet/spreadsheet_workbook.xlsx', index=False)

print("Spreadsheet workbook created.")

NameError: name 'pd' is not defined